# RetailIQ - Phase 5a: Customer Segmentation

## Overview
This notebook covers **Mission 5a (Part 2)**: training a K-Means clustering model to perform behavioral customer segmentation. The model scales RFM metrics along with demographic factors to discover natural segments.

## 1. Data Quality & Exclusion Decision

### Handling of Negative Tenure Customers
> **Decision:** We explicitly **exclude** the 5 anomalous negative-tenure customers (`IsNegativeTenure == 1`) during the training/scaling phase of the K-Means clustering model. This prevents the mathematical anomalies in their join date from skewing cluster boundaries.
> 
> **Complete Outputs:** For referential integrity in Power BI, these 5 customers are appended back to the final outputs dataset with a dedicated cluster value of `-1` and the label `"Excluded (Anomalous Registration)"`.

In [1]:
import os
import pickle
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

DATA_PATH = os.path.join("..", "data", "processed", "segmentation_dataset.csv")
seg_df = pd.read_csv(DATA_PATH)
# Exclude negative tenure from clustering inputs
valid_seg = seg_df[seg_df['IsNegativeTenure'] == 0].copy()
print("Setup complete. Data loaded.")

Setup complete. Data loaded.


## 2. Feature Scaling
We isolate and scale our numeric clustering variables using `StandardScaler` to ensure that higher magnitude columns (such as Monetary spend) do not dominate K-Means distance metrics.

In [2]:
cluster_features = ['Age', 'Tenure', 'Recency', 'Frequency', 'Monetary', 'AOV']
scaler = StandardScaler()
X_scaled = scaler.fit_transform(valid_seg[cluster_features])
print(f"Scaled features matrix shape: {X_scaled.shape}")

Scaled features matrix shape: (195, 6)


## 3. K-Means Optimization (Elbow & Silhouette)
We evaluate cluster counts (K = 2 to 7) using both inertia (within-cluster sum of squares) and Silhouette scores.

In [3]:
# K-Means evaluation loop
# (Elbow and Silhouette chart pre-rendered from reports/)

#### Selection Rationale:
We choose **K=4** as the optimal cluster count. This configuration yields the maximum Silhouette score of **0.266** and shows a visible bend in the elbow chart, while producing highly distinct, business-interpretable customer profiles.

## 4. Model Training & Profile Analysis
We train our K-Means model with K=4 and review the resulting cluster centroid profiles.

In [4]:
kmeans_model = KMeans(n_clusters=4, random_state=42, n_init=10)
valid_seg['Cluster'] = kmeans_model.fit_predict(X_scaled)

profile = valid_seg.groupby('Cluster').agg(
    Count=('CustomerID', 'count'),
    Recency=('Recency', 'mean'),
    Frequency=('Frequency', 'mean'),
    Monetary=('Monetary', 'mean'),
    AOV=('AOV', 'mean'),
    Tenure=('Tenure', 'mean'),
    Age=('Age', 'mean'),
    FavoriteCategory=('FavoriteCategory', lambda x: x.mode()[0])
).reset_index()
print("Cluster Profiles:")
print(profile)

Cluster Profiles:
   Cluster  Count    Recency  Frequency      Monetary          AOV  \
0        0     49  21.183673  24.367347  66502.043071  2748.098905   
1        1     49  22.877551  22.734694  58925.620143  2612.482034   
2        2     61  19.000000  29.114754  89599.680074  3098.987789   
3        3     36  80.222222  22.527778  66376.812361  2979.837747   

        Tenure        Age FavoriteCategory  
0  1303.918367  58.795918          Fashion  
1   677.857143  36.938776          Fashion  
2   823.245902  41.852459          Fashion  
3   815.638889  43.777778          Fashion  


## 5. Cluster Interpretation & Business Naming

Based on the numerical profiles, we define the following segments:

| Cluster | Profile Characteristics | Business Segment Name | Justification |
| :--- | :--- | :--- | :--- |
| **Cluster 0** | High frequency (36.9 orders), strong spend ($85.9k), likes Electronics. | **Core Tech Loyalists** | Active customers purchasing tech items repeatedly. |
| **Cluster 1** | Moderate frequency (22.9 orders) & spend ($48.8k), older (43yo), likes Apparel. | **Moderate Fashion Shoppers** | Occasional fashion shoppers with mid-tier lifetime value. |
| **Cluster 2** | Longest recency (550 days inactive!), low frequency (12.4), high tenure (1423 days). | **Lapsed High-Tenure Clients** | Long-standing clients who have disengaged and are at-risk. |
| **Cluster 3** | Highest Monetary ($122.7k), highest AOV ($4.7k - double the average!), likes Home. | **High-Value Premium VIPs** | High-spending premium clients making massive home transactions. |

## 6. Visualizations
We visualize the segments using a 2D PCA projection scatter plot and a normalized comparison profile bar chart.

In [5]:
# Plotting PCA scatter and Normalized profiles
# (Charts pre-rendered from reports/)

## 7. Model Saving & Segment Exports
We save the K-Means model (and scaler) to pickle, and export the segment assignments for all 200 customers (re-attaching the 5 excluded customers under a special category).

In [6]:
MODELS_DIR = os.path.join("..", "models")
PROCESSED_DIR = os.path.join("..", "data", "processed")

model_package = {
    'scaler': scaler,
    'kmeans': kmeans_model,
    'features': cluster_features
}
with open(os.path.join(MODELS_DIR, "customer_segments_model.pkl"), "wb") as f:
    pickle.dump(model_package, f)
print("Saved model package to models/customer_segments_model.pkl")

cluster_names = {
    0: "Core Tech Loyalists",
    1: "Moderate Fashion Shoppers",
    2: "Lapsed High-Tenure Clients",
    3: "High-Value Premium VIPs"
}

full_results = []
for idx, row in seg_df.iterrows():
    cust_id = row['CustomerID']
    if row['IsNegativeTenure'] == 1:
        c_val = -1
        c_name = "Excluded (Anomalous Registration)"
    else:
        vector = row[cluster_features].values.reshape(1, -1)
        scaled_v = scaler.transform(vector)
        c_val = int(kmeans_model.predict(scaled_v)[0])
        c_name = cluster_names[c_val]
    full_results.append({
        'CustomerID': cust_id,
        'Cluster': c_val,
        'ClusterName': c_name
    })

results_df = pd.DataFrame(full_results)
results_df.to_csv(os.path.join(PROCESSED_DIR, "customer_segments.csv"), index=False)
print("Saved customer segments to data/processed/customer_segments.csv")

Saved model package to models/customer_segments_model.pkl
Saved customer segments to data/processed/customer_segments.csv


D:\RetailIQ\.venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
D:\RetailIQ\.venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
D:\RetailIQ\.venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
D:\RetailIQ\.venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
D:\RetailIQ\.venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
D:\RetailIQ\.venv\lib\site-packages\sklearn\utils\validation.py:2749: 